# Chapter 17 — Limited Dependent Variables
## University of Neuchâtel — Python for Econometrics

Topics:
1. Linear Probability Model (LPM)
2. Logit model — log-odds and marginal effects
3. Probit model — marginal effects
4. Comparing LPM, Logit, Probit
5. Tobit model for censored data

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import norm, logistic
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Dataset: `mroz`

In [ ]:
mroz = woo.dataWoo('mroz')
print('Shape:', mroz.shape)
print(mroz[['inlf','educ','exper','kidslt6','nwifeinc']].head(10))

In [ ]:
print('Dependent variable: inlf (in labour force)')
print(mroz['inlf'].value_counts())
print(f'Participation rate: {mroz["inlf"].mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
mroz['hours'].hist(bins=20, ax=axes[0], edgecolor='black')
axes[0].set_title('Hours worked (censored at 0)')
axes[0].set_xlabel('Hours')
mroz['educ'].hist(bins=15, ax=axes[1], edgecolor='black', color='orange')
axes[1].set_title('Years of Education')
axes[1].set_xlabel('Years')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch17_01_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2 — Linear Probability Model (LPM)

**Model:** $P(inlf=1|x) = \beta_0 + \beta_1 educ + \beta_2 exper + \beta_3 kidslt6 + \beta_4 nwifeinc$

OLS applied to a binary dependent variable.

$\hat{\beta}_j$ = change in **probability** of participating for a unit increase in $x_j$.

**Limitation:** Predicted probabilities can fall outside $[0,1]$.

### Step 2.1 — Manual OLS (LPM)

In [ ]:
n = len(mroz)
X_lpm = np.column_stack([
    np.ones(n),
    mroz['educ'].values,
    mroz['exper'].values,
    mroz['kidslt6'].values,
    mroz['nwifeinc'].values
])
y_lpm = mroz['inlf'].values.astype(float)
beta_lpm = np.linalg.solve(X_lpm.T @ X_lpm, X_lpm.T @ y_lpm)
labels_lpm = ['const','educ','exper','kidslt6','nwifeinc']
print('LPM (manual):')
for lbl, b in zip(labels_lpm, beta_lpm):
    print(f'  {lbl:12s}: {b:.5f}')

### Step 2.2 — Statsmodels LPM

In [ ]:
res_lpm = smf.ols('inlf ~ educ + exper + kidslt6 + nwifeinc', data=mroz).fit()
print(res_lpm.summary())

### Step 2.3 — Compare

In [ ]:
sm_lpm = res_lpm.params
print('LPM comparison:')
for lbl, b_man in zip(labels_lpm, beta_lpm):
    match = np.isclose(b_man, sm_lpm[lbl])
    print(f'  {lbl:12s}: manual={b_man:.5f}, sm={sm_lpm[lbl]:.5f}, match={match}')

### Step 2.4 — Check for out-of-range probabilities

In [ ]:
y_hat_lpm = res_lpm.fittedvalues
print(f'Min predicted prob: {y_hat_lpm.min():.3f}')
print(f'Max predicted prob: {y_hat_lpm.max():.3f}')
n_neg = (y_hat_lpm < 0).sum()
n_gt1 = (y_hat_lpm > 1).sum()
print(f'Predicted < 0: {n_neg} ({100*n_neg/len(mroz):.1f}%)')
print(f'Predicted > 1: {n_gt1} ({100*n_gt1/len(mroz):.1f}%)')

## 3 — Logit Model

**Model:** $P(y=1|x) = \Lambda(x\beta) = \frac{e^{x\beta}}{1+e^{x\beta}}$

The **logistic CDF** $\Lambda(\cdot)$ maps $x\beta \in (-\infty,+\infty)$ to $(0,1)$.

**Marginal effect at the mean:**

$$\frac{\partial P}{\partial x_j} = \Lambda(\bar{x}\hat{\beta})[1 - \Lambda(\bar{x}\hat{\beta})] \cdot \hat{\beta}_j$$

### Step 3.1 — Fit Logit

In [ ]:
res_logit = smf.logit('inlf ~ educ + exper + kidslt6 + nwifeinc', data=mroz).fit()
print(res_logit.summary())

### Step 3.2 — Marginal effects manually (at means)

In [ ]:
x_bar = np.array([1,
    mroz['educ'].mean(),
    mroz['exper'].mean(),
    mroz['kidslt6'].mean(),
    mroz['nwifeinc'].mean()
])
b_logit = res_logit.params.values
xb = x_bar @ b_logit
Lambda_xb = logistic.cdf(xb)  # = 1/(1+exp(-xb))
scale = Lambda_xb * (1 - Lambda_xb)
me_logit_manual = scale * b_logit[1:]  # exclude constant
print(f'Lambda(xbar*b) = {Lambda_xb:.4f}')
print(f'Scale factor   = {scale:.4f}')
print('Marginal effects (manual):')
for lbl, me in zip(labels_lpm[1:], me_logit_manual):
    print(f'  {lbl:12s}: {me:.5f}')

### Step 3.3 — Compare with statsmodels get_margeff

In [ ]:
mfx_logit = res_logit.get_margeff(at='mean')
print(mfx_logit.summary())
print('Marginal effect comparison (at mean):')
for lbl, me_man, me_sm in zip(labels_lpm[1:], me_logit_manual, mfx_logit.margeff):
    match = np.isclose(me_man, me_sm)
    print(f'  {lbl:12s}: manual={me_man:.5f}, sm={me_sm:.5f}, match={match}')

## 4 — Probit Model

**Model:** $P(y=1|x) = \Phi(x\beta)$ where $\Phi$ is the standard normal CDF.

**Marginal effect at the mean:**

$$\frac{\partial P}{\partial x_j} = \phi(\bar{x}\hat{\beta}) \cdot \hat{\beta}_j$$

where $\phi$ is the standard normal PDF.

### Step 4.1 — Fit Probit

In [ ]:
res_probit = smf.probit('inlf ~ educ + exper + kidslt6 + nwifeinc', data=mroz).fit()
print(res_probit.summary())

### Step 4.2 — Marginal effects manually (at means)

In [ ]:
b_probit = res_probit.params.values
xb_probit = x_bar @ b_probit
phi_xb = norm.pdf(xb_probit)  # standard normal PDF
me_probit_manual = phi_xb * b_probit[1:]
print(f'phi(xbar*b) = {phi_xb:.4f}')
print('Probit marginal effects (manual):')
for lbl, me in zip(labels_lpm[1:], me_probit_manual):
    print(f'  {lbl:12s}: {me:.5f}')

### Step 4.3 — Compare with statsmodels

In [ ]:
mfx_probit = res_probit.get_margeff(at='mean')
print('Probit marginal effect comparison (at mean):')
for lbl, me_man, me_sm in zip(labels_lpm[1:], me_probit_manual, mfx_probit.margeff):
    match = np.isclose(me_man, me_sm)
    print(f'  {lbl:12s}: manual={me_man:.5f}, sm={me_sm:.5f}, match={match}')

## 5 — Compare LPM, Logit, Probit

In [ ]:
educ_range = np.linspace(mroz['educ'].min(), mroz['educ'].max(), 100)
mean_exper = mroz['exper'].mean()
mean_kids  = mroz['kidslt6'].mean()
mean_nwi   = mroz['nwifeinc'].mean()
b_lpm    = res_lpm.params.values
b_log    = res_logit.params.values
b_prob   = res_probit.params.values
xb_lpm  = lambda e: b_lpm[0]+b_lpm[1]*e+b_lpm[2]*mean_exper+b_lpm[3]*mean_kids+b_lpm[4]*mean_nwi
xb_log  = lambda e: b_log[0]+b_log[1]*e+b_log[2]*mean_exper+b_log[3]*mean_kids+b_log[4]*mean_nwi
xb_prob = lambda e: b_prob[0]+b_prob[1]*e+b_prob[2]*mean_exper+b_prob[3]*mean_kids+b_prob[4]*mean_nwi
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(educ_range, [xb_lpm(e) for e in educ_range], label='LPM', linewidth=2)
ax.plot(educ_range, [logistic.cdf(xb_log(e)) for e in educ_range], label='Logit', linewidth=2)
ax.plot(educ_range, [norm.cdf(xb_prob(e)) for e in educ_range], label='Probit', linewidth=2, linestyle='--')
ax.set_xlabel('Years of Education')
ax.set_ylabel('P(inlf=1)')
ax.set_title('Predicted Probability of Labor Force Participation')
ax.legend()
ax.axhline(0, color='gray', linestyle=':')
ax.axhline(1, color='gray', linestyle=':')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch17_02_lpm_logit_probit.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6 — Summary: Marginal Effects Comparison

In [ ]:
sumtab = pd.DataFrame({
    'Variable': labels_lpm[1:],
    'LPM':    [res_lpm.params[l] for l in labels_lpm[1:]],
    'Logit':  list(mfx_logit.margeff),
    'Probit': list(mfx_probit.margeff)
})
print(sumtab.round(5).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch17_marginal_effects.csv'), index=False)
print('Saved table_ch17_marginal_effects.csv')